# Phase 5 — Yocto build adapter (REAL component versions)

Turns a Yocto image manifest into component facts, then loads them with the deployment context and reasons. Versions are the real Poky Scarthgap 5.0.8 + meta-uthp pins — no more made-up seed data.
Swap in a real `<image>.rootfs.manifest` from a build and this notebook is unchanged.

In [1]:
import sys; sys.path.append('../src')
import build_adapter, ssc_graph as kg

out, n = build_adapter.generate('../data/uthp_image.rootfs.manifest', '../data/uthp_build.ttl')
print(f'Adapter wrote {n} triples to {out}')

Adapter wrote 101 triples to ../data/uthp_build.ttl


In [2]:
g = kg.load(['../data/uthp_build.ttl', '../data/uthp_context.ttl'])
print(len(g), 'triples (build + context + inference)')

1033 triples (build + context + inference)


In [3]:
# Real components + versions now in the graph
for f in ['../queries/cq01_components_in_images.rq', '../queries/cq10_may_impact_function.rq']:
    print(f'\n== {f.split("/")[-1]} ==')
    for row in kg.run_file(g, f):
        print('  ', ' | '.join(kg.short(x) for x in row))


== cq01_components_in_images.rq ==
   uthp_scarthgap_image | pkg_bash_5_2_21 | 5.2.21
   uthp_scarthgap_image | pkg_busybox_1_36_1 | 1.36.1
   uthp_scarthgap_image | pkg_curl_8_7_1 | 8.7.1
   uthp_scarthgap_image | pkg_dropbear_2022_83 | 2022.83
   uthp_scarthgap_image | pkg_expat_2_6_4 | 2.6.4
   uthp_scarthgap_image | pkg_glibc_2_39 | 2.39
   uthp_scarthgap_image | pkg_libgcrypt_1_10_3 | 1.10.3
   uthp_scarthgap_image | pkg_libxml2_2_12_10 | 2.12.10
   uthp_scarthgap_image | pkg_openssh_9_6p1 | 9.6p1
   uthp_scarthgap_image | pkg_openssl_3_2_4 | 3.2.4
   uthp_scarthgap_image | pkg_python3_pretty_j1939_0_0_0 | 0.0.0
   uthp_scarthgap_image | pkg_python3_sae_j1939_2_0_12 | 2.0.12
   uthp_scarthgap_image | pkg_systemd_255_17 | 255.17
   uthp_scarthgap_image | pkg_truckdevil_0_0 | 0.0
   uthp_scarthgap_image | pkg_util_linux_2_39_3 | 2.39.3
   uthp_scarthgap_image | pkg_zlib_1_3_1 | 1.3.1

== cq10_may_impact_function.rq ==
   pkg_bash_5_2_21 | braking
   pkg_busybox_1_36_1 | braking
   

## What changed
- The graph now holds **real** UTHP component versions (openssl 3.2.4, zlib 1.3.1, ...), each with a purl identifier and build provenance.
- CQ10 still fires: every component on the gateway is inferred to *may impact braking* via the network pivot.
- **No vulnerability facts yet** — CQ7/CQ12 are empty. That is correct: cybersecurity intelligence (CVE/CWE/VEX) is **Phase 6**, which will match these real versions against real advisories.

**Next (Phase 6):** the intelligence adapter — pull CVE/CWE/VEX and correlate to these versions (e.g., confirm zlib 1.3.1 is *not* affected by CVE-2022-37434 — the false-positive case).